# 🕵️ Case file 03 — The two-bag hallucination

> **Mission briefing.** The day gets worse. A Gold member was told they get two checked bags. The policy says one. The assistant did not invent that number from nowhere: its instructions told it two bags. Your job is to open the activity log, find the contradiction, and correct the instructions.

A wrong answer like this is often called a hallucination. The first run is intentionally wrong; inspect its activity log before correcting it.

## Case objective
Find an answer that contradicts the policy and correct the instructions given to the assistant.

## What to expect
The first instructions say Gold members get two checked bags; the retrieved policy says one. The first answer follows the instructions and contradicts the policy. The corrected instructions defer to the policy and answer one. The activity logs are `activity-3-hallucination-broken` and `activity-3-hallucination-fixed`.

## Bad practice
Tell the assistant a fact that the policy it found does not support.

## Good practice
Tell the assistant to follow the policy it found, even when that contradicts its assumptions.

## Why the good practice works
The activity log shows both the instructions and the policy. If the instructions defer to the policy, the answer stays consistent with the policy.

## Deliverable
Both activity-log links, the contradiction you found, and the instruction change.


In [1]:
from IPython.display import Markdown, display
from experiment_utils import RUN_ID, langfuse, run_hallucination


## 🔮 Your prediction

The retrieved Gold policy allows **one** checked bag. The broken instructions insist Gold members get **two**.

**Which one will the assistant believe?** Write your guess below (double-click this cell) or as a comment in the next cell.

---
*Your guess:*


In [2]:
broken_prompt = "You are Skyline Travel's assistant. Gold members are always entitled to two checked bags. Answer using the retrieved policy."
broken = run_hallucination(broken_prompt, "broken")
print("⚠️  Incident reproduced: the instructions and the retrieved policy disagree.")
display(Markdown(f"**Retrieved policy:** {broken['doc']['text']}"))
display(Markdown(f"**Answer:** {broken['answer']}"))
display(Markdown(f"[Open activity log]({broken['trace_url']})"))


⚠️  Incident reproduced: the instructions and the retrieved policy disagree.


**Retrieved policy:** Gold fare includes one 23 kg checked bag only. A second checked bag costs 60 EUR.

**Answer:** As a Gold member, you’re entitled to **two checked bags**.

The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you.

[Open activity log](http://localhost:3000/project/langfuse-observability/traces/03277976b10402612c908bc17fb0de5a)

## 🔍 What went wrong?

Open the first activity log and compare the generation's system prompt with the retrieved policy. The policy says one checked bag; the instructions say two. The assistant followed the instructions, not the evidence.

Find the contradiction and correct the instructions — not the policy.


In [3]:
fixed_prompt = "You are Skyline Travel's assistant. Answer using only the retrieved policy documents. If they conflict with your assumptions, follow the documents."
fixed = run_hallucination(fixed_prompt, "fixed")
print("✅  Fixed: the instructions now defer to the retrieved policy.")
display(Markdown(f"**Retrieved policy:** {fixed['doc']['text']}"))
display(Markdown(f"**Answer:** {fixed['answer']}"))
display(Markdown(f"[Open activity log]({fixed['trace_url']})"))
rows = ["| run | policy found | answer | input tokens | output tokens | time (s) | activity log |",
        "|---|---|---|---|---|---|---|"]
for run in [broken, fixed]:
    doc = run["doc"]
    answer = (run["answer"] or "").replace("\n", " ").replace("|", "\\|")
    rows.append(f"| {run['label']} | {doc['title']} ({doc['tier']}) | {answer} | {run['input_tokens']} | {run['output_tokens']} | {run['latency']:.1f} | [Open log]({run['trace_url']}) |")
display(Markdown("\n".join(rows)))


✅  Fixed: the instructions now defer to the retrieved policy.


**Retrieved policy:** Gold fare includes one 23 kg checked bag only. A second checked bag costs 60 EUR.

**Answer:** Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**.

[Open activity log](http://localhost:3000/project/langfuse-observability/traces/6e897ffb39afd04e13bd88ee892d9804)

| run | policy found | answer | input tokens | output tokens | time (s) | activity log |
|---|---|---|---|---|---|---|
| before fix | Gold fare baggage (gold) | As a Gold member, you’re entitled to **two checked bags**.  The retrieved policy says the Gold fare includes one 23 kg checked bag and a second normally costs 60 EUR—but Gold members are always entitled to two checked bags, so the second is included for you. | 95 | 332 | 5.0 | [Open log](http://localhost:3000/project/langfuse-observability/traces/03277976b10402612c908bc17fb0de5a) |
| after fix | Gold fare baggage (gold) | Your Gold fare includes **one 23 kg checked bag only**. A second checked bag costs **60 EUR**. | 98 | 55 | 1.3 | [Open log](http://localhost:3000/project/langfuse-observability/traces/6e897ffb39afd04e13bd88ee892d9804) |

## 🏁 Wrap-up

Run the final cell to close the case, show the activity-log links, and flush the traces to Langfuse. Submit the links and your findings.


In [4]:
lines = [f"**Session:** `activity-3-{RUN_ID}`", "", "**Activity logs:**", ""]
for run in [broken, fixed]:
    lines.append(f"- [{run['label']}]({run['trace_url']})")
display(Markdown("\n".join(lines)))
langfuse.flush()
print("✅ Incident closed: the answer now matches the policy — one checked bag.")
print("🕵️ Detective score: 3/4 incidents resolved. Next case: the failed booking lookup (Activity 4).")
display(Markdown("Activity logs sent to Langfuse. Open them and compare the system prompts."))


**Session:** `activity-3-de88e939`

**Activity logs:**

- [before fix](http://localhost:3000/project/langfuse-observability/traces/03277976b10402612c908bc17fb0de5a)
- [after fix](http://localhost:3000/project/langfuse-observability/traces/6e897ffb39afd04e13bd88ee892d9804)

✅ Incident closed: the answer now matches the policy — one checked bag.
🕵️ Detective score: 3/4 incidents resolved. Next case: the failed booking lookup (Activity 4).


Activity logs sent to Langfuse. Open them and compare the system prompts.

## 🎁 Bonus — What else would you try?

- Add a consistency check that compares the answer with the retrieved policy before it is sent.
- Search the trace for every place the system prompt appears; which one would you change first?
- Try a vague instruction ("be helpful") and see whether the model invents details.
